In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (SparkSession.builder
    .appName("SahlPay-Partie3-DF")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.executor.memory", "1g")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate())
spark.sparkContext.setLogLevel("WARN")

tx_schema = StructType([
    StructField("tx_id",        StringType(),    False),
    StructField("type",         StringType(),    True),
    StructField("payer_wallet", StringType(),    True),
    StructField("payee_id",     StringType(),    True),
    StructField("device_id",    StringType(),    True),
    StructField("city",         StringType(),    True),
    StructField("channel",      StringType(),    True),
    StructField("status",       StringType(),    True),
    StructField("amount_mad",   DoubleType(),    True),
    StructField("event_time",   TimestampType(), True),
])
merchant_schema = StructType([
    StructField("merchant_id", StringType(), False),
    StructField("name",        StringType(), True),
    StructField("city",        StringType(), True),
    StructField("category",    StringType(), True),
    StructField("latitude",    DoubleType(), True),
    StructField("longitude",   DoubleType(), True),
])

tx        = spark.read.schema(tx_schema).json("s3a://lake/bronze/history/transactions_history.jsonl")
merchants = spark.read.schema(merchant_schema).json("s3a://lake/reference/merchants.json")

tx.printSchema()
print("Transactions :", tx.count(), "| Commerçants :", merchants.count())
tx.show(3, truncate=False)

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a3e9be42-a451-412d-bb16-a884fae18928;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 546ms :: artifacts dl 13ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|

root
 |-- tx_id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- payer_wallet: string (nullable = true)
 |-- payee_id: string (nullable = true)
 |-- device_id: string (nullable = true)
 |-- city: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- status: string (nullable = true)
 |-- amount_mad: double (nullable = true)
 |-- event_time: timestamp (nullable = true)



Transactions : 50000 | Commerçants : 300
+-----------+----+------------+--------+---------+------+-------+---------+----------+-------------------+
|tx_id      |type|payer_wallet|payee_id|device_id|city  |channel|status   |amount_mad|event_time         |
+-----------+----+------------+--------+---------+------+-------+---------+----------+-------------------+
|TX-00000001|P2M |W-10481     |M-100   |D-c9dc72 |Rabat |QR     |INITIATED|71.09     |2026-11-14 08:00:02|
|TX-00000002|P2M |W-10972     |M-261   |D-0ed2ec |Meknes|APP    |INITIATED|182.07    |2026-11-14 08:00:04|
|TX-00000003|P2M |W-10032     |M-226   |D-8d3aed |Agadir|USSD   |INITIATED|11.13     |2026-11-14 08:00:04|
+-----------+----+------------+--------+---------+------+-------+---------+----------+-------------------+
only showing top 3 rows



In [4]:
def pipeline_df():
    comp = tx.filter(F.col("status") == "COMPLETED").dropDuplicates(["tx_id"])
    total = comp.groupBy("city").agg(F.sum("amount_mad").alias("total")).collect()
    moy   = comp.groupBy("city").agg(F.avg("amount_mad").alias("moyenne")).collect()
    top   = (comp.filter(F.col("type") == "P2M")
                 .groupBy("payee_id").count()
                 .orderBy(F.desc("count")).limit(5).collect())
    return total, moy, top

for i in range(3):
    t = time.time()
    pipeline_df()
    print(f"Essai DF {i+1} : {time.time() - t:.1f} s")

Essai DF 1 : 11.7 s


Essai DF 2 : 5.2 s


[Stage 55:======================================>                   (2 + 1) / 3]

Essai DF 3 : 4.5 s


In [5]:
comp = tx.filter(F.col("status") == "COMPLETED").dropDuplicates(["tx_id"])
(comp.groupBy("city")
     .agg(F.round(F.sum("amount_mad"), 2).alias("total_mad"),
          F.count("*").alias("nb"),
          F.round(F.avg("amount_mad"), 2).alias("moyenne_mad"))
     .orderBy(F.desc("total_mad"))
     .show())

[Stage 61:======================================>                   (2 + 1) / 3]

+----------+---------+----+-----------+
|      city|total_mad|  nb|moyenne_mad|
+----------+---------+----+-----------+
|Casablanca|969822.08|3740|     259.31|
|     Rabat|577769.98|2180|     265.03|
| Marrakech| 501856.7|1950|     257.36|
|       Fes| 409119.3|1520|     269.16|
|    Tanger|383106.01|1511|     253.54|
|    Agadir|333253.03|1274|     261.58|
|    Meknes|262511.07|1016|     258.38|
|   Tetouan|227248.18| 766|     296.67|
|   Kenitra|198912.47| 758|     262.42|
|     Oujda|157724.07| 686|     229.92|
+----------+---------+----+-----------+



In [8]:
tx.createOrReplaceTempView("transactions")

spark.sql("""
    WITH c AS (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY tx_id ORDER BY event_time) AS rn
        FROM transactions
        WHERE status = 'COMPLETED'
    )
    SELECT city,
           ROUND(SUM(amount_mad), 2) AS total_mad,
           COUNT(*)                  AS nb,
           ROUND(AVG(amount_mad), 2) AS moyenne_mad
    FROM c
    WHERE rn = 1
    GROUP BY city
    ORDER BY total_mad DESC
""").show()

[Stage 80:===================>                                      (1 + 2) / 3]

+----------+---------+----+-----------+
|      city|total_mad|  nb|moyenne_mad|
+----------+---------+----+-----------+
|Casablanca|969822.08|3740|     259.31|
|     Rabat|577769.98|2180|     265.03|
| Marrakech| 501856.7|1950|     257.36|
|       Fes| 409119.3|1520|     269.16|
|    Tanger|383106.01|1511|     253.54|
|    Agadir|333253.03|1274|     261.58|
|    Meknes|262511.07|1016|     258.38|
|   Tetouan|227248.18| 766|     296.67|
|   Kenitra|198912.47| 758|     262.42|
|     Oujda|157724.07| 686|     229.92|
+----------+---------+----+-----------+



In [7]:
m = merchants.withColumnRenamed("city", "m_city")
p2m = comp.filter(F.col("type") == "P2M")

jointure = p2m.join(m, p2m.payee_id == m.merchant_id, "inner")

res = (jointure.groupBy("category", "m_city")
       .agg(F.round(F.sum("amount_mad"), 2).alias("total_mad"),
            F.count("*").alias("nb"))
       .orderBy("m_city", F.desc("total_mad")))
res.show(25, truncate=False)

[Stage 76:>                                                         (0 + 1) / 1]

+------------+----------+---------+---+
|category    |m_city    |total_mad|nb |
+------------+----------+---------+---+
|epicerie    |Agadir    |109433.51|450|
|restauration|Agadir    |51938.84 |250|
|e-commerce  |Agadir    |35272.69 |146|
|pharmacie   |Agadir    |18061.96 |73 |
|carburant   |Agadir    |14399.63 |43 |
|epicerie    |Casablanca|200682.5 |826|
|e-commerce  |Casablanca|145912.42|569|
|restauration|Casablanca|136090.42|534|
|carburant   |Casablanca|84774.0  |268|
|pharmacie   |Casablanca|66615.05 |286|
|telecom     |Casablanca|61704.32 |318|
|restauration|Fes       |116933.6 |435|
|epicerie    |Fes       |103775.3 |384|
|carburant   |Fes       |34544.59 |148|
|pharmacie   |Fes       |14200.08 |57 |
|telecom     |Fes       |12995.02 |63 |
|e-commerce  |Fes       |4461.52  |22 |
|restauration|Kenitra   |67431.38 |314|
|epicerie    |Kenitra   |29151.48 |113|
|telecom     |Kenitra   |22434.77 |71 |
|e-commerce  |Kenitra   |5480.45  |16 |
|carburant   |Kenitra   |4670.96  |20 |


In [9]:
from pyspark.sql.functions import udf

# --- Version UDF Python ---
def amount_band_py(a):
    if a is None:
        return None
    if a < 50:
        return "micro"
    if a <= 2000:
        return "standard"
    return "eleve"

amount_band_udf = udf(amount_band_py, StringType())
spark.udf.register("amount_band", amount_band_py, StringType())     # utilisable aussi en SQL

# --- Version sans UDF (expressions natives de Spark) ---
band_expr = (F.when(F.col("amount_mad").isNull(), F.lit(None))
              .when(F.col("amount_mad") < 50, "micro")
              .when(F.col("amount_mad") <= 2000, "standard")
              .otherwise("eleve"))

base = tx.cache()
base.count()                                                        # remplit le cache

def run_udf():
    return base.withColumn("band", amount_band_udf("amount_mad")).groupBy("band").count().collect()

def run_native():
    return base.withColumn("band", band_expr).groupBy("band").count().collect()

for nom, f in [("UDF", run_udf), ("when/otherwise", run_native)]:
    for i in range(3):
        t = time.time(); f()
        print(f"{nom:15s} essai {i+1} : {time.time() - t:.2f} s")

print(sorted(run_udf()))
print(sorted(run_native()))

UDF             essai 1 : 2.50 s
UDF             essai 2 : 0.66 s
UDF             essai 3 : 0.99 s
when/otherwise  essai 1 : 0.79 s
when/otherwise  essai 2 : 0.62 s
when/otherwise  essai 3 : 0.49 s
[Row(band='eleve', count=431), Row(band='micro', count=9577), Row(band='standard', count=39992)]
[Row(band='eleve', count=431), Row(band='micro', count=9577), Row(band='standard', count=39992)]


In [10]:
clean = (tx.dropDuplicates(["tx_id", "status"])
           .withColumn("amount_band", band_expr))
print("Avant :", tx.count(), "| après dédoublonnage :", clean.count())

(clean.repartition("city")
      .write.mode("overwrite")
      .partitionBy("city")
      .parquet("s3a://lake/silver/transactions_history/"))

back = spark.read.parquet("s3a://lake/silver/transactions_history/")
print("Relu :", back.count(), "lignes")
back.printSchema()
back.groupBy("city").count().orderBy("city").show()

Avant : 50000 | après dédoublonnage : 49063


Relu : 49063 lignes
root
 |-- tx_id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- payer_wallet: string (nullable = true)
 |-- payee_id: string (nullable = true)
 |-- device_id: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- status: string (nullable = true)
 |-- amount_mad: double (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- amount_band: string (nullable = true)
 |-- city: string (nullable = true)



[Stage 133:==========================================>              (3 + 1) / 4]

+----------+-----+
|      city|count|
+----------+-----+
|    Agadir| 4021|
|Casablanca|11844|
|       Fes| 4903|
|   Kenitra| 2403|
| Marrakech| 6229|
|    Meknes| 3242|
|     Oujda| 2170|
|     Rabat| 6994|
|    Tanger| 4829|
|   Tetouan| 2428|
+----------+-----+



In [11]:
import boto3
from botocore.client import Config

s3 = boto3.client("s3", endpoint_url="http://minio:9000",
    aws_access_key_id="minioadmin", aws_secret_access_key="minioadmin123",
    region_name="us-east-1",
    config=Config(signature_version="s3v4", s3={"addressing_style": "path"}))

res = s3.list_objects_v2(Bucket="lake", Prefix="silver/transactions_history/", Delimiter="/")
for p in res.get("CommonPrefixes", []):
    print(p["Prefix"])

silver/transactions_history/city=Agadir/
silver/transactions_history/city=Casablanca/
silver/transactions_history/city=Fes/
silver/transactions_history/city=Kenitra/
silver/transactions_history/city=Marrakech/
silver/transactions_history/city=Meknes/
silver/transactions_history/city=Oujda/
silver/transactions_history/city=Rabat/
silver/transactions_history/city=Tanger/
silver/transactions_history/city=Tetouan/


In [12]:
jointure.explain(True)

== Parsed Logical Plan ==
Join Inner, (payee_id#3 = merchant_id#20)
:- Filter (type#1 = P2M)
:  +- Deduplicate [tx_id#0]
:     +- Filter (status#7 = COMPLETED)
:        +- Relation [tx_id#0,type#1,payer_wallet#2,payee_id#3,device_id#4,city#5,channel#6,status#7,amount_mad#8,event_time#9] json
+- Project [merchant_id#20, name#21, city#22 AS m_city#846, category#23, latitude#24, longitude#25]
   +- Relation [merchant_id#20,name#21,city#22,category#23,latitude#24,longitude#25] json

== Analyzed Logical Plan ==
tx_id: string, type: string, payer_wallet: string, payee_id: string, device_id: string, city: string, channel: string, status: string, amount_mad: double, event_time: timestamp, merchant_id: string, name: string, m_city: string, category: string, latitude: double, longitude: double
Join Inner, (payee_id#3 = merchant_id#20)
:- Filter (type#1 = P2M)
:  +- Deduplicate [tx_id#0]
:     +- Filter (status#7 = COMPLETED)
:        +- Relation [tx_id#0,type#1,payer_wallet#2,payee_id#3,device_i

In [13]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")      # désactive le broadcast
spark.read.schema(tx_schema).json("s3a://lake/bronze/history/transactions_history.jsonl") \
     .filter("status = 'COMPLETED' AND type = 'P2M'") \
     .join(m, F.col("payee_id") == F.col("merchant_id")).explain()
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10485760") # remet la valeur par défaut (10 Mo)

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- SortMergeJoin [payee_id#4831], [merchant_id#20], Inner
   :- Sort [payee_id#4831 ASC NULLS FIRST], false, 0
   :  +- Exchange hashpartitioning(payee_id#4831, 4), ENSURE_REQUIREMENTS, [plan_id=3316]
   :     +- Filter ((((isnotnull(status#4835) AND isnotnull(type#4829)) AND (status#4835 = COMPLETED)) AND (type#4829 = P2M)) AND isnotnull(payee_id#4831))
   :        +- InMemoryTableScan [tx_id#4828, type#4829, payer_wallet#4830, payee_id#4831, device_id#4832, city#4833, channel#4834, status#4835, amount_mad#4836, event_time#4837], [isnotnull(status#4835), isnotnull(type#4829), (status#4835 = COMPLETED), (type#4829 = P2M), isnotnull(payee_id#4831)]
   :              +- InMemoryRelation [tx_id#4828, type#4829, payer_wallet#4830, payee_id#4831, device_id#4832, city#4833, channel#4834, status#4835, amount_mad#4836, event_time#4837], StorageLevel(disk, memory, deserialized, 1 replicas)
   :                    +- FileScan json [tx_id#0,